In [1043]:
from openpyxl import load_workbook
from collections import namedtuple
import pandas as pd
import re
from pathlib import Path
from pprint import pprint

In [ ]:
p = Path(r"C:\Users\Heikki\OneDrive - HJT-RAK Oy\001 TOIMISTOURAKAT\\")
files = [x for x in p.rglob('001 Laskentapohja*')]

In [1035]:
wb = load_workbook(filename="001 Laskentapohja, 2024.xlsx", data_only=True)

In [1036]:
wb.sheetnames

['Tuntihinnoittelut',
 'Materiaalihinnoittelut',
 'Purkutyöt',
 'Väliseinät sisällä',
 'Maalaukset',
 'Alakatot',
 'Laatoitukset',
 'Lattiat',
 'Oviasennukset ja listoitukset',
 'Erillishankinnat',
 'Eritelysivu',
 'Koontisivu',
 'Massalista, rak. tarvikkeet ',
 'Päivitykset pohjaan']

In [1063]:
# tuntihinnoittelut
lines = namedtuple('item', ['kuvaus', 'tunnit'])
rows = [lines(*row[1:]) for row in wb['Tuntihinnoittelut'].iter_rows(values_only=True) if row.count(None) < 2]
pd.DataFrame(rows).iloc[2:, :].to_dict(orient='records')


[{'kuvaus': 'Työmaan perustus + työmaataulu ja ilmoitukset', 'tunnit': 16},
 {'kuvaus': 'Suojaukset', 'tunnit': 16},
 {'kuvaus': 'Vaneriseinän rakennus', 'tunnit': 0},
 {'kuvaus': 'wc alakaton purku ja uudelleen rakennus', 'tunnit': 30},
 {'kuvaus': 'Haalaukset', 'tunnit': 40},
 {'kuvaus': 'IV-tulppaukset', 'tunnit': 0},
 {'kuvaus': 'Kaihdinten kunnon tarkastus ja huolto', 'tunnit': 0},
 {'kuvaus': 'Apurungon rakennus', 'tunnit': 16},
 {'kuvaus': 'Puulasiovien maalaus', 'tunnit': 34},
 {'kuvaus': 'Listojen maalaus', 'tunnit': 16},
 {'kuvaus': 'Teräslasioven siirto', 'tunnit': 8},
 {'kuvaus': 'Ovien lyhennyksiä', 'tunnit': 8},
 {'kuvaus': 'Yhteensä', 'tunnit': 184},
 {'kuvaus': 'Hinta', 'tunnit': 7728}]

In [ ]:
# materiaalihinnoittelu

line = namedtuple('item', ['POS', 'Tuote', 'Hinta', 'Kuvaus', 'kpl', 'Yht'])

a = [line(*row[:6]) for row in wb['Materiaalihinnoittelut'].iter_rows(values_only=True)]
b = [row for row in a if isinstance(row.Yht, (int, float)) and not row.Yht == 0 and not row.Tuote == 'Yhteensä']

In [887]:
# erittelysivu

line1 = namedtuple('item', ['selite', 'materiaalit', 'työt'])
line2 = namedtuple('item', ['Erillishankinnat', 'Hinta', 
                            'Toimittaja_urakoitsija', 'ignore1', 
                            'Toimitus', 'ignore2', 'ignore3'])

erittelyt_cols_left = []
erittelyt_cols_right = []
erillishankinnat_cols = []
kokonaishinnat = []

temp = []

for i, row in enumerate(wb['Eritelysivu'].iter_rows(values_only=True)):
    if i < 77:
        erittelyt_cols_left.append(line1(*row[:3]))
        erittelyt_cols_right.append(line1(*row[4:7]))
    if 77 <= i < 97:
        erittelyt_cols_left.append(line1(*row[:3]))
    if 97 <= i < 114:
        erillishankinnat_cols.append(line2(*row))
    if i > 114:
        if row.count(None) < 5:
            if len(temp) == 0:
                ntuple_names = list(row)
                ntuple_names = [f'id_{i}' if x is None else x for i, x in enumerate(ntuple_names)]
                ntuple_names = [x.replace('-', '_') for x in ntuple_names]
                ntuple = namedtuple('item', ntuple_names)
                temp.append(ntuple)
                continue
            if len(temp) != 0:
                out = temp.pop()
                out = out(*row)
                kokonaishinnat.append(out)


df_erillishankinnat = pd.DataFrame(erillishankinnat_cols).iloc[:, [0, 1,4]]

df_left = pd.DataFrame(erittelyt_cols_left)
df_right = pd.DataFrame(erittelyt_cols_right)
df_right.columns = df_left.columns
df_erittelyt = pd.concat([df_left, df_right])

# TODO: tarvitaanko kokonaishinnat jos on jo vastaavat erittelysivulla?
[a._asdict() for a in kokonaishinnat]


[{'Omakustannehinta': None,
  'Työt': 45487.3,
  'Materiaalit': 66004.51108,
  'id_3': None,
  'Erillishankinnat': 49294,
  'id_5': None,
  'id_6': None},
 {'Kate_osuus': None,
  'Työ': 6663.405882352942,
  'Materiaalit': 11519.750321307189,
  'id_3': None,
  'Erillishankinnat': 8698.94117647059,
  'id_5': None,
  'id_6': None}]

In [890]:
# koontisivu

line = namedtuple('item', ['selite', 'ignore', 'hinta', 'selite2'])
a = [line(*row) for row in wb['Koontisivu'].iter_rows(values_only=True) if row.count(None) < 3]
b = [x for x in a if isinstance(x.hinta, (int, float))]
c = [x for x in b if (isinstance(x.selite, str) or isinstance(x.selite2, str)) and x.selite != 'Yhteensä']

df = pd.DataFrame(c)
df.iloc[:, [0,2,3]].dropna()


,selite,hinta,selite2
2,0,7058.823529,Siivous
3,0,2.352941,LVI-urakka / sähkö ei tule meiltä
4,0,27058.823529,Lasiseinät
5,ST Team,15294.117647,Lasisiirtoseinät
6,0,1764.705882,Heloitukset
7,0,6117.647059,Ovet
8,0,588.235294,Timanttiporaus


In [891]:
# purkutyöt

line = namedtuple('item', ['a', 'b', 'c'])
lines = [line(*row[:3]) for row in wb['Purkutyöt'].iter_rows(values_only=True) if row.count(None) < 6]

df = pd.DataFrame(lines).iloc[:, 1:].dropna(thresh=2)
df = df[df.iloc[:, 0] != 'Tuntiveloitushinta']

In [892]:
# väliseinät sisällä

line = namedtuple('item', ['a', 'b', 'c'])
lines = [line(*row[:3]) for row in wb['Väliseinät sisällä'].iter_rows(values_only=True)]
df = pd.DataFrame(lines)
df = df[((df.iloc[:, 0].astype(str).str.contains('Hinta', na=False)) |
        (df.iloc[:, 0].astype(str).str.contains('Työ', na=False))) & 
        (pd.to_numeric(df.iloc[:, 2], errors='coerce') != 0)]

In [1066]:
# väliseinät sisällä

line = namedtuple("Line", ["työ", "hinta"])

lines = [
    line(*row[:2])
    for row in wb["Väliseinät sisällä"].iter_rows(values_only=True)
]

pos_blocks = {}
test_blocks = []
current_pos = None

for line in lines:
    # Uusi POS
    if isinstance(line.työ, str) and re.fullmatch(r"POS-\d+", line.työ.strip()):
        current_pos = line.työ.strip()
        pos_blocks[current_pos] = []
        continue

    # ei mitään ennen ekaa POS
    if current_pos is None:
        continue

    # rivien filtteröinti
    if (
        line.työ is not None
        and isinstance(line.hinta, (int, float))
        and line.hinta > 0
    ):
        pos_blocks[current_pos].append(line._asdict())
        test_blocks.append(line._asdict())

df = pd.DataFrame(test_blocks)
df

,työ,hinta
0,2-kertanen,95
1,Runkokiinnikkeet,3
2,Hobarit,3
3,Pientarvikkeet,300
4,1-kertanen,60
5,Runkokiinnikkeet,2
6,Hobarit,2
7,Pientarvikkeet,150
8,1-kertainen,20
9,Runkokiinnikkeet,1


In [1067]:
# maalaukset

line = namedtuple("Line", ["työ", "hinta"])

lines = [
    line(*row[:2])
    for row in wb["Maalaukset"].iter_rows(values_only=True)
]

pos_blocks = {}
test_blocks = []
current_pos = None

for line in lines:
    # Uusi POS
    if isinstance(line.työ, str) and re.fullmatch(r"POS-\d+", line.työ.strip()):
        current_pos = line.työ.strip()
        pos_blocks[current_pos] = []
        continue

    # ei mitään ennen ekaa POS
    if current_pos is None:
        continue

    # rivien filtteröinti
    if (
        line.työ is not None
        and isinstance(line.hinta, (int, float))
        and line.hinta > 0
    ):
        pos_blocks[current_pos].append(line._asdict())
        test_blocks.append(line._asdict())

df = pd.DataFrame(test_blocks)
df

,työ,hinta
0,"Osittainen tasoitus, LF",300.00
1,Ylitasoitus,300.00
2,Pohjamaalaus 2X,1200.00
3,"Pintamaalaus 2X, sävytetty",1200.00
4,"Pientarvikkeet, €",400.00
5,"Hinta, materiaalit",6919.12
6,Työ,8748.00
7,Yht.,15667.12
8,Ylitasoitus,200.00
9,"Rappauskorjaus, 10mm asti",100.00


In [894]:
# alakatot

line = namedtuple("Line", ["työ", "hinta"])

lines = [
    line(*row[:2])
    for row in wb["Alakatot"].iter_rows(values_only=True)
]

pos_blocks = {}
test_blocks = []
current_pos = None

for line in lines:
    # Uusi POS
    if isinstance(line.työ, str) and re.fullmatch(r"POS-\d+", line.työ.strip()):
        current_pos = line.työ.strip()
        pos_blocks[current_pos] = []
        continue

    # ei mitään ennen ekaa POS
    if current_pos is None:
        continue

    # rivien filtteröinti
    if (
        line.työ is not None
        and isinstance(line.hinta, (int, float))
        and line.hinta > 0
    ):
        pos_blocks[current_pos].append(line._asdict())
        test_blocks.append(line._asdict())

df = pd.DataFrame(test_blocks)
df = df[~df.iloc[:, 0].astype(str).str.contains('Lisä')]

In [896]:
# laatoitukset

# alakatot

line = namedtuple("Line", ["työ", "hinta"])

lines = [
    line(*row[:2])
    for row in wb["Laatoitukset"].iter_rows(values_only=True)
]

pos_blocks = {}
test_blocks = []
current_pos = None

for line in lines:
    # Uusi POS
    if isinstance(line.työ, str) and re.fullmatch(r"POS-\d+", line.työ.strip()):
        current_pos = line.työ.strip()
        pos_blocks[current_pos] = []
        continue

    # ei mitään ennen ekaa POS
    if current_pos is None:
        continue

    # rivien filtteröinti
    if (
        line.työ is not None
        and isinstance(line.hinta, (int, float))
        and line.hinta > 0
    ):
        pos_blocks[current_pos].append(line._asdict())
        test_blocks.append(line._asdict())

df = pd.DataFrame(test_blocks)
df = df[~df.iloc[:, 0].astype(str).str.contains('Lisä')]

In [898]:
# laatoitukset

# alakatot

line = namedtuple("Line", ["työ", "hinta"])

lines = [
    line(*row[:2])
    for row in wb["Lattiat"].iter_rows(values_only=True)
]

pos_blocks = {}
test_blocks = []
current_pos = None

for line in lines:
    # Uusi POS
    if isinstance(line.työ, str) and re.fullmatch(r"POS-\d+", line.työ.strip()):
        current_pos = line.työ.strip()
        pos_blocks[current_pos] = []
        continue

    # ei mitään ennen ekaa POS
    if current_pos is None:
        continue

    # rivien filtteröinti
    if (
        line.työ is not None
        and isinstance(line.hinta, (int, float))
        and line.hinta > 0
    ):
        pos_blocks[current_pos].append(line._asdict())
        test_blocks.append(line._asdict())

df = pd.DataFrame(test_blocks)
df = df[~df.iloc[:, 0].astype(str).str.contains('Lisä')]

In [919]:
wb.sheetnames

['Tuntihinnoittelut',
 'Materiaalihinnoittelut',
 'Purkutyöt',
 'Väliseinät sisällä',
 'Maalaukset',
 'Alakatot',
 'Laatoitukset',
 'Lattiat',
 'Oviasennukset ja listoitukset',
 'Erillishankinnat',
 'Eritelysivu',
 'Koontisivu',
 'Massalista, rak. tarvikkeet ',
 'Päivitykset pohjaan']

In [920]:
# oviasennukset ja listoitukset

# alakatot

line = namedtuple("Line", ["työ", "hinta"])

lines = [
    line(*row[:2])
    for row in wb["Oviasennukset ja listoitukset"].iter_rows(values_only=True)
]

pos_blocks = {}
test_blocks = []
current_pos = None

for line in lines:
    # Uusi POS
    if isinstance(line.työ, str) and re.fullmatch(r"POS-\d+", line.työ.strip()):
        current_pos = line.työ.strip()
        pos_blocks[current_pos] = []
        continue

    # ei mitään ennen ekaa POS
    if current_pos is None:
        continue

    # rivien filtteröinti
    if (
        line.työ is not None
        and isinstance(line.hinta, (int, float))
        and line.hinta > 0
    ):
        pos_blocks[current_pos].append(line._asdict())
        test_blocks.append(line._asdict())

df = pd.DataFrame(test_blocks)
df = df[~df.iloc[:, 0].astype(str).str.contains('Lisä')]

In [1034]:
# erillishankinat

line = namedtuple('item', ['selite', 'hinta'])

lines = [row[1:3] for row in wb['Erillishankinnat'].iter_rows(
    values_only=True
    )
   if str(row[1:3][0]) not in ('Kate-%', 'Kate-osuus')
   and row[1:3].count(None) < 1
   ][1:]

lines = [line for line in lines if line[1] > 30]

item = [x[0] for x in lines[::2]]
hinta = [x[1] for x in lines[1::2]]

lines = [x for x in zip(item, hinta)]
lines = [line(*x) for x in lines]

pd.DataFrame(lines)

,selite,hinta
0,Siivous,7058.823529
1,Lasiseinät,27058.823529
2,Lasisiirtoseinät,15294.117647
3,Heloitukset,1764.705882
4,Ovet,6117.647059
5,Timanttiporaus,588.235294
